# Feature selection for y_ny
L1-penalized logistic regression ("lasso for classification"): like ridge, but the L1 penalty pushes useless features' weights to exactly 0.
C (inverse penalty strength) is picked by time-series cross-validation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score

# AUC vs number of features, adding them in order of |L1 weight| (most important first)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [ ]:
# --- config: point this at any cleaned dataset ---
DATA   = "../data/final_data_ny_big.feather"
TARGET = "y_ny"        # binary 0/1 label
DROP   = ["ret_ny"]    # columns that leak the label (missing ones are ignored)
OUT    = "../data/less_features.feather"

data = pd.read_feather(DATA).sort_index()
# flatten MultiIndex columns like ("stock_ret", "NXPI") -> "stock_ret_NXPI"; flat names stay as they are
data.columns = ["_".join(filter(None, map(str, c))) if isinstance(c, tuple) else str(c) for c in data.columns]
X = data.drop(columns=[TARGET, *DROP], errors="ignore").select_dtypes("number")
X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="all").ffill().dropna()
y = data[TARGET].loc[X.index].dropna().astype(int)
X = X.loc[y.index]

# chronological 80/20 split, same as 5_Model_training
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print("train:", X_train.shape, "test:", X_test.shape, "| share of up days (train):", round(y_train.mean(), 3))


In [ ]:
# TimeSeriesSplit: each fold trains on the past and validates on the next block, no shuffling (no look-ahead)
model = make_pipeline(
    StandardScaler(),
    LogisticRegressionCV(
        Cs=np.logspace(-3, 1, 30),   # small C = strong penalty = fewer features
        l1_ratios=[1.0],             # pure L1 (lasso)
        solver="saga",
        cv=TimeSeriesSplit(n_splits=5),
        scoring="roc_auc",
        max_iter=10000,
    ),
).fit(X_train, y_train)

In [ ]:
lr = model[-1]
cv_auc = lr.scores_[1].mean(axis=0).max()
p_test = model.predict_proba(X_test)[:, 1]
print(f"best C: {lr.C_[0]:.4f}")
print(f"CV AUC:   {cv_auc:.3f}  ")
print(f"test AUC: {roc_auc_score(y_test, p_test):.3f}")
print(f"test acc: {accuracy_score(y_test, p_test > 0.5):.3f}   vs always-up baseline {y_test.mean():.3f}")

In [ ]:
# features kept by L1 (non-zero weight), on standardized scale so sizes are comparable
coef = pd.Series(lr.coef_[0], index=X.columns)
selected = coef[coef != 0].sort_values(key=abs, ascending=False)
assert len(selected), "L1 set every weight to 0: no feature helps at any C on this dataset"
print(f"{len(selected)} / {len(coef)} features kept")

selected.iloc[::-1].plot.barh(figsize=(6, max(2, 0.3 * len(selected))))
plt.axvline(0, c="grey"); plt.xlabel("weight (+ = pushes toward up day)")
plt.show()
selected

In [ ]:

ranked = selected.index
ks = range(1, len(ranked) + 1)
cv_aucs, test_aucs = [], []
for k in ks:
    cols = ranked[:k]
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000))
    cv_aucs.append(cross_val_score(m, X_train[cols], y_train, cv=TimeSeriesSplit(n_splits=5), scoring="roc_auc").mean())
    test_aucs.append(roc_auc_score(y_test, m.fit(X_train[cols], y_train).predict_proba(X_test[cols])[:, 1]))

plt.figure(figsize=(8, 4))
plt.plot(ks, cv_aucs, "o-", label="CV AUC (train)")
plt.plot(ks, test_aucs, "s-", label="test AUC")
plt.axhline(0.5, c="grey", ls="--", label="random")
plt.xlabel("number of features (top-k by |L1 weight|)"); plt.ylabel("AUC")
plt.legend(); plt.grid(alpha=0.3); plt.show()
print("best k (CV):", ks[int(np.argmax(cv_aucs))])

In [ ]:
# Understanding the top features
TOP = 10
top = selected.index[:TOP]
fig, axes = plt.subplots(1, 3, figsize=(30, 10))

# 1) weight stability: refit the L1 model (best C) on each time-series fold
#    a useful feature keeps the same sign and a similar size in every fold
fold_coefs = []
for tr, _ in TimeSeriesSplit(n_splits=5).split(X_train):
    m = make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1.0, solver="saga", C=lr.C_[0], max_iter=10000))
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    fold_coefs.append(pd.Series(m[-1].coef_[0], index=X.columns)[top])
fold_coefs = pd.DataFrame(fold_coefs)
axes[0].boxplot(fold_coefs[top[::-1]].values, vert=False, tick_labels=top[::-1])
axes[0].axvline(0, c="grey"); axes[0].set_title("L1 weight across CV folds (stability)")
axes[0].set_xlabel("weight")

# 2) univariate AUC of each feature alone, train vs test (0.5 = useless, <0.5 = negative relation)
uni = pd.DataFrame({
    "train": [roc_auc_score(y_train, X_train[c]) for c in top],
    "test":  [roc_auc_score(y_test,  X_test[c])  for c in top],
}, index=top)
uni[::-1].plot.barh(ax=axes[1], left=0.5)  # bars start at 0.5 so direction is visible
axes[1].axvline(0.5, c="grey"); axes[1].set_title("AUC of each feature alone")
axes[1].set_xlabel("AUC")

# 3) correlation between top features: highly correlated pairs carry the same information
corr = X_train[top].corr()
im = axes[2].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_xticks(range(len(top)), top, rotation=90); axes[2].set_yticks(range(len(top)), top)
axes[2].set_title("correlation between top features (train)")
fig.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout(); plt.show()
uni.assign(weight=selected[top], sign_consistent=(np.sign(fold_coefs) == np.sign(selected[top])).mean().values)

## Choosing criteria : 

Stable sign. Keep features with sign_consistent >= MIN_SIGN (default 0.8, so at least 4 of 5 folds). Features below it flip direction from fold to fold, so drop them. One caveat: your folds use an expanding window and overlap, so 5/5 is weaker evidence than it looks.

No duplicates. If two kept features have |corr| > 0.5 with each other, keep only the more important one. Your heatmap shows which pairs these are.

How many to keep. The best k from the curve is just the highest point of a noisy curve. Use the 1-SE rule instead: take the smallest k whose CV AUC is within one standard error of the best. That usually leaves far fewer features and overfits less.

Optional sanity check. The feature's AUC on its own in train should fall on the same side of 0.5 as its weight's sign. Disagreement can be a real effect that only shows up alongside other features, but it's a reason to look at that feature more closely.


In [ ]:
tbl = uni.assign(weight=selected[top],sign_consistent=(np.sign(fold_coefs) == np.sign(selected[top])).mean().values)
MIN_SIGN = 0.8  # share of folds where the weight keeps its sign (an L1 zero counts as a flip)
keep = tbl.index[tbl.sign_consistent >= MIN_SIGN].tolist()
assert keep, f"no feature has sign_consistent >= {MIN_SIGN}: lower MIN_SIGN or raise TOP"

# drop near-duplicates, walking in importance order
c = X_train[keep].corr().abs()
final = []
for f in keep:
    if all(c.loc[f, g] < 0.5 for g in final):
        final.append(f)

m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000))
cv = cross_val_score(m, X_train[final], y_train, cv=TimeSeriesSplit(5), scoring="roc_auc")
print(len(final), "features | CV AUC", cv.mean().round(3), "±", cv.std().round(3))
print(final)


In [ ]:
data_less_features = X[final]
data_less_features

In [ ]:
assert data_less_features.shape[1], "no features selected, not overwriting " + OUT
data_less_features.to_feather(OUT)